In [ ]:

from google.colab import files
import pandas as pd

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name, encoding="latin1")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Saving SampleSuperstore.csv to SampleSuperstore (1).csv
Dataset loaded successfully!
Dataset shape: (9994, 13)

First 5 rows:


,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164



Column names:
['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']

Missing values:
Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64


In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Find the target column
target_col = next(
    (c for c in df.columns if c.strip().lower() == "sales"),
    None
)

if target_col is None:
    raise ValueError(
        f"Sales column not found. Available columns: {df.columns.tolist()}"
    )

# 2. Clean target values
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[target_col]).copy()

# 3. Remove columns that identify records/customers/products
# and Profit, which is calculated separately from Sales
exclude = [
    "row id", "order id", "customer id", "customer name",
    "product id", "product name", "profit"
]

drop_cols = [
    c for c in df.columns
    if c != target_col and c.strip().lower() in exclude
]

X = df.drop(columns=[target_col] + drop_cols).copy()
y = df[target_col].copy()

# 4. Convert date columns into year/month features
for col in list(X.columns):
    if "date" in col.strip().lower():
        dates = pd.to_datetime(X[col], errors="coerce")
        X[col + "_year"] = dates.dt.year
        X[col + "_month"] = dates.dt.month
        X = X.drop(columns=[col])

# Remove columns that are entirely empty
X = X.dropna(axis=1, how="all")

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# 5. Preprocessing
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# 6. Split dataset: 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Total records:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Input features:", X.shape[1])

# 7. Define the two algorithms
models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting Regression": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

# 8. Train, predict and evaluate
results = []
trained_models = {}

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("regressor", algorithm)
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[name] = model

    print("\nModel:", name)
    print(f"MAE:      {mae:.4f}")
    print(f"MSE:      {mse:.4f}")
    print(f"RMSE:     {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")

# 9. Display comparison table
results_df = pd.DataFrame(results)

print("\nFINAL MODEL COMPARISON")
display(results_df.round(4))

# 10. Show actual and predicted sales
best_model_name = results_df.sort_values(
    by="RMSE"
).iloc[0]["Model"]

print("\nLowest RMSE model:", best_model_name)

best_model = trained_models[best_model_name]
sample_predictions = best_model.predict(X_test)

prediction_df = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": sample_predictions
})

print("\nSample predictions:")
display(prediction_df.head(10))

Total records: 9994
Training records: 7995
Testing records: 1999
Input features: 11

Model: Linear Regression
MAE:      242.4177
MSE:      486525.3473
RMSE:     697.5137
R2 Score: 0.1764

Model: Gradient Boosting Regression
MAE:      200.5197
MSE:      470681.8153
RMSE:     686.0625
R2 Score: 0.2032

FINAL MODEL COMPARISON


,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,242.4177,486525.3473,697.5137,0.1764
1,Gradient Boosting Regression,200.5197,470681.8153,686.0625,0.2032



Lowest RMSE model: Gradient Boosting Regression

Sample predictions:


,Actual Sales,Predicted Sales
0,563.808,221.529931
1,36.672,55.061937
2,37.300,67.221197
3,212.058,340.881415
4,171.288,549.818560
5,99.136,52.457570
6,340.182,228.917378
7,17.380,169.006464
8,22.620,-19.517184
9,71.632,159.730279
